# Libraries

In [17]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold

# Data

## Classification

In [2]:
breast_cancer = fetch_ucirepo(id=17)

bcwd_df = pd.concat(
    [breast_cancer.data.features, breast_cancer.data.targets],
    axis=1
)
bcwd_df.head()

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


In [3]:
adult = fetch_ucirepo(id=2)
adult_df = pd.concat(
    [adult.data.features, adult.data.targets],
    axis=1
)
adult_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [6]:
higgs_df = pd.read_csv(
    "HIGGS.csv.gz",
    compression="gzip",
    header=None
)

higgs_df

,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,1.0,0.869293,-0.635082,0.225690,0.327470,-0.689993,0.754202,-0.248573,-1.092064,0.000000,...,-0.010455,-0.045767,3.101961,1.353760,0.979563,0.978076,0.920005,0.721657,0.988751,0.876678
1,1.0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,...,-1.138930,-0.000819,0.000000,0.302220,0.833048,0.985700,0.978098,0.779732,0.992356,0.798343
2,1.0,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,...,1.128848,0.900461,0.000000,0.909753,1.108330,0.985692,0.951331,0.803252,0.865924,0.780118
3,0.0,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,...,-0.678379,-1.360356,0.000000,0.946652,1.028704,0.998656,0.728281,0.869200,1.026736,0.957904
4,1.0,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,...,-0.373566,0.113041,0.000000,0.755856,1.361057,0.986610,0.838085,1.133295,0.872245,0.808487
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10999995,1.0,1.159912,1.013847,0.108615,1.495524,-0.537545,2.342396,-0.839740,1.320683,0.000000,...,-0.097068,1.190680,3.101961,0.822136,0.766772,1.002191,1.061233,0.837004,0.860472,0.772484
10999996,1.0,0.618388,-1.012982,1.110139,0.941023,-0.379199,1.004656,0.348535,-1.678593,2.173076,...,-0.216995,1.049177,3.101961,0.826829,0.989809,1.029104,1.199679,0.891481,0.938490,0.865269
10999997,1.0,0.700559,0.774251,1.520182,0.847112,0.211230,1.095531,0.052457,0.024553,2.173076,...,1.585235,1.713962,0.000000,0.337374,0.845208,0.987610,0.883422,1.888438,1.153766,0.931279
10999998,0.0,1.178030,0.117796,-1.276980,1.864457,-0.584370,0.998519,-1.264549,1.276333,0.000000,...,1.399515,-1.313189,0.000000,0.838842,0.882890,1.201380,0.939216,0.339705,0.759070,0.719119


## Regression

In [ ]:
california = fetch_california_housing(as_frame=True)
california_df = california.frame
california_df.head()

In [ ]:
concrete_compressive_strength = fetch_ucirepo(id=165) 
ccs_df = pd.concat(
    [concrete_compressive_strength.data.features, concrete_compressive_strength.data.targets],
    axis=1
)
ccs_df.head()

In [ ]:
seoul_bike_sharing_demand = fetch_ucirepo(id=560)
sbsd_df = pd.concat(
    [seoul_bike_sharing_demand.data.features, seoul_bike_sharing_demand.data.targets],
    axis=1
)
sbsd_df.head()

# Models

In [21]:
X = breast_cancer.data.features
y = (breast_cancer.data.targets).iloc[:, 0].map({"B": 0, "M": 1}).astype("int32")

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_pool)
X_test_scaled = scaler.transform(X_test)

model = tf.keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.F1Score(average='macro')]
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                      │ (None, 32)                  │             992 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,025 (4.00 KB)

 Trainable params: 1,025 (4.00 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model.fit(
    X_train_scaled,
    y_pool,
    epochs=100,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.6373 - auc: 0.6285 - f1_score: 0.5211 - loss: 0.6633 - val_accuracy: 0.7971 - val_auc: 0.8992 - val_f1_score: 0.6602 - val_loss: 0.5100
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8731 - auc: 0.9409 - f1_score: 0.5211 - loss: 0.4240 - val_accuracy: 0.9420 - val_auc: 0.9773 - val_f1_score: 0.6602 - val_loss: 0.3563
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9275 - auc: 0.9729 - f1_score: 0.5211 - loss: 0.3134 - val_accuracy: 0.9565 - val_auc: 0.9929 - val_f1_score: 0.6602 - val_loss: 0.2694
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9404 - auc: 0.9809 - f1_score: 0.5211 - loss: 0.2547 - val_accuracy: 0.9710 - val_auc: 0.9992 - val_f1_score: 0.6602 - val_loss: 0.2164
Epoch 5/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9456 - auc: 0.9851 - f1_score: 0.5211 - loss: 0.2193 - val_accuracy: 0.9710 - val_auc: 0.9992 - val_f1_score: 0.6602 - val_los

In [24]:
y_prob = model.predict(X_test_scaled)
y_pred = (y_prob >= 0.5).astype(int)

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='macro')

print("Test accuracy:", test_acc)
print("Test AUC:", test_auc)
print("Test F1:", test_f1)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Test accuracy: 0.9912280701754386
Test AUC: 0.9880952380952381
Test F1: 0.9905276277523889


In [26]:
X_labelled, X_unlabelled, y_labelled, y_unlabelled = train_test_split(
    X_pool,
    y_pool,
    test_size=0.9,
    stratify=y_pool,
    random_state=100
)

indices = np.arange(len(X_pool))
X_labelled, X_unlabelled, y_labelled, y_unlabelled, \
idx_labelled, idx_unlabelled = train_test_split(
    X_pool,
    y_pool,
    indices,
    test_size=0.9,
    stratify=y_pool,
    random_state=100
)

idx_unlabelled

array([451,  55,  74, 150,  49,  99, 232, 299, 375, 426,  30, 107, 254,
        12, 300, 248, 307, 364, 367, 266, 339, 105, 338, 419, 118, 194,
       124, 390, 383, 264, 260, 322, 253, 433, 137, 394, 351, 135, 358,
        47, 186, 110,  66,  78, 316, 296,  51,  33,   5, 305, 145, 140,
        40, 142,  11, 321, 443, 198, 417, 453,  41, 246, 372, 151,  80,
       278, 146, 442, 354, 201, 149, 143, 447, 249, 411, 301, 251, 448,
        89, 247, 312,  63, 424,  53, 229, 340, 405,  60, 352, 220,   4,
       377, 344, 429, 192, 223, 177, 114,  68, 415, 113, 384, 207,  10,
       369,  57, 123, 430,  76, 418,  18, 191, 187, 120,  25, 170, 225,
       450, 436, 242, 125,  17, 357, 213,  69, 404,  52, 423,  61,  23,
       298,  64, 119,  20, 432, 102, 128, 361, 283,  46, 255, 106, 346,
       269, 289, 413, 116, 319,  19, 100, 126,  75, 313, 265,  44, 336,
         1, 444, 211, 315, 403, 353, 288, 360, 221, 441, 323, 337, 452,
        59, 341,   8, 258, 273,  85, 262, 174,  43, 408,  36, 28

In [31]:
def passive_sampling(model, X_unlabelled, n_samples):
    selected_indices = np.random.choice(
        X_unlabelled.shape[0],
        size=n_samples,
        replace=False
    )
    return selected_indices

def uncertainty_sampling(model, X_unlabelled, n_samples):
    probabilities = model.predict(
        X_unlabelled,
        verbose=0
    ).ravel()
    selected_indices = np.argsort(np.abs(probabilities - 0.5))[:n_samples]
    return selected_indices

def label_data(selected_indices, labelled, unlabelled):
    selected_pool_indices = unlabelled[selected_indices]
    new_labelled = np.concatenate([labelled, selected_pool_indices], axis=0)
    new_unlabelled = np.setdiff1d(
        unlabelled,
        selected_pool_indices
    )

    return new_labelled, new_unlabelled

    # X_labelled = X_pool[new_labelled]
    # y_labelled = y_pool[new_labelled]
    # X_unlabelled = X_pool[new_unlabelled]
    # y_unlabelled = y_pool[new_unlabelled]

    # return X_labelled, X_unlabelled, y_labelled, y_unlabelled


def active_learning(model, X_pool, y_pool, X_test, y_test, sampling, num_iter=10, unlab_size=0.9, random_state=100, verbose=False):
    indices = np.arange(len(X_pool))
    X_labelled, X_unlabelled, y_labelled, y_unlabelled, \
    idx_labelled, idx_unlabelled = train_test_split(
        X_pool,
        y_pool,
        indices,
        test_size=unlab_size,
        stratify=y_pool,
        random_state=random_state
    )

    n_samples = len(idx_unlabelled) // (num_iter-1)
    history = []

    for i in range(num_iter):
        if i == num_iter - 2:
            n_samples = idx_unlabelled.shape[0]

        # update pool of labelled data
        X_labelled = X_pool.iloc[idx_labelled]
        y_labelled = y_pool.iloc[idx_labelled]
        X_unlabelled = X_pool.iloc[idx_unlabelled]
        y_unlabelled = y_pool.iloc[idx_unlabelled]
        # X_labelled = X_pool[idx_labelled]
        # y_labelled = y_pool[idx_labelled]
        # X_unlabelled = X_pool[idx_unlabelled]
        # y_unlabelled = y_pool[idx_unlabelled]

        # train model on labelled data
        model.fit(
            X_labelled,
            y_labelled,
            epochs=100,
            batch_size=32,
            validation_split=0.15,
            verbose=0
        )

         # Evaluate
        y_prob = model.predict(X_test, verbose=0).ravel()
        y_pred = (y_prob >= 0.5).astype(int)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        auc = roc_auc_score(y_test, y_prob)

        if verbose:
            print(f"Iteration {i+1}: {idx_labelled.shape[0]} labelled instances\n"
                  f"\tAccuracy = {acc}\n"
                  f"\tmacro F1 = {f1}\n"
                  f"\tAUC = {auc}\n"
            )
        history.append({'Accuracy': acc, 'macro F1': f1, 'ROC-AUC': auc})

        # determine instances that should be labelled
        if i < num_iter - 1:
            selected_indices = sampling(model, X_unlabelled, n_samples)
            idx_labelled, idx_unlabelled = label_data(selected_indices, idx_labelled, idx_unlabelled)

    return history

In [32]:
active_model = tf.keras.Sequential([
    layers.Input(shape=(X_pool.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation='sigmoid')
])

active_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.F1Score(average='macro')]
)

history = active_learning(
    active_model,
    X_pool,
    y_pool,
    X_test,
    y_test,
    passive_sampling,
    num_iter=3,
    unlab_size=0.9,
    random_state=100,
    verbose=True
)

Iteration 1: 45 labelled instances
	Accuracy = 0.7894736842105263
	macro F1 = 0.7285714285714285
	AUC = 0.7820767195767196

Iteration 2: 250 labelled instances
	Accuracy = 0.7894736842105263
	macro F1 = 0.7285714285714285
	AUC = 0.7642195767195767

Iteration 3: 455 labelled instances
	Accuracy = 0.9122807017543859
	macro F1 = 0.9013157894736843
	AUC = 0.9371693121693122

